[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C12/E12-4_Pannes_Parc_CORRIGE.ipynb)

# Panne dans les vingt-quatre heures : anticiper à partir des derniers relevés

## Le contexte

Seize machines d'un parc sont relevées heure par heure : charge, température de fonctionnement, vibration, heures écoulées depuis la dernière révision. Un indicateur `panne` vaut 1 quand une machine tombe en panne pendant l'heure ; elle est alors remise à neuf. L'équipe de maintenance ne veut pas constater les pannes, mais les prévenir : quand une alerte annonce une panne dans les vingt-quatre heures, elle intervient avant.

Une fausse alerte coûte une intervention inutile ; une panne manquée coûte un arrêt non planifié. Les pannes sont rares : 68 sur 24 000 relevés. La classe à prévoir, « une panne arrive dans les vingt-quatre heures », reste minoritaire ; c'est un problème de classification déséquilibrée sur des séries temporelles.

Défi : prédire, pour chaque relevé, si une panne surviendra dans les 24 heures suivantes, et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Le F-score macro est la moyenne des F-scores des deux classes : il pèse autant la classe rare que la classe fréquente. Le modèle de départ est une régression logistique à poids de classes équilibrés, sur les mesures du relevé, les heures depuis la révision et la présence d'une panne dans les vingt-quatre heures écoulées ; il atteint 0,700. Deux règles fixées à la main, sans apprentissage, servent de comparaison : une alarme dès que la vibration dépasse 8 mm/s (0,695) et une alarme dès que la révision a plus de 240 heures (0,564). À vous de faire mieux. La base de test est formée, pour chaque machine, du dernier quart de ses relevés moins ses 24 derniers, dont l'avenir n'est pas observé : jamais un tirage au hasard, car le futur ne doit pas passer dans l'apprentissage.

## Les données

Fichier `MecaIA-Pannes_Parc.xlsx`, dans le dossier `donnees` du dépôt : 24 000 relevés, 7 colonnes. Une ligne est le relevé horaire d'une machine.

| Colonne | Signification | Unité |
|---|---|---|
| `horodatage` | heure pleine du relevé, 1 500 relevés par machine | date et heure |
| `machine` | identifiant de la machine, m01 à m16 | catégorie |
| `charge_pct` | charge de la machine pendant l'heure (quelques cases vides) | % |
| `temperature_C` | température de fonctionnement (quelques cases vides) | °C |
| `vibration_mm_s` | vitesse de vibration efficace | mm/s |
| `heures_depuis_revision` | heures écoulées depuis la dernière remise à neuf | h |
| `panne` | 1 si la machine tombe en panne pendant l'heure | 0 ou 1 |

Origine : jeu simulé, un dommage cumulé croît avec la charge, fait monter la vibration et la température, et déclenche la panne quand il atteint un seuil ; chaque machine a sa vibration saine et sa vitesse d'usure. La révision préventive recommandée est de 300 heures. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, le pas de temps, le nombre de relevés par machine et le nombre de pannes : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [1]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Pannes_Parc.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Pannes_Parc.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

df["horodatage"] = pd.to_datetime(df["horodatage"])
vides = df.isna().sum()
print("dimensions :", df.shape)
print("cases vides par colonne :")
print(vides[vides > 0].to_string())
pas = df.groupby("machine")["horodatage"].diff().dropna()
print("pas de temps :", {str(k): int(n) for k, n in pas.value_counts().items()})
print("relevés par machine :", sorted(df["machine"].value_counts().unique().tolist()))
pannes_par_machine = df.groupby("machine")["panne"].sum()
print("pannes par machine : de", int(pannes_par_machine.min()), "à", int(pannes_par_machine.max()), "; au total", int(df["panne"].sum()))

dimensions : (24000, 7)
cases vides par colonne :
charge_pct       741
temperature_C    494
pas de temps : {'0 days 01:00:00': 23984}
relevés par machine : [1500]
pannes par machine : de 1 à 8 ; au total 68


Lecture du résultat : 24 000 relevés, 16 machines de 1 500 relevés chacune, au pas régulier d'une heure. Deux colonnes ont des cases vides : la charge (741) et la température (494). Les pannes sont rares, 0,3 % des relevés : l'exactitude d'un modèle qui ne signalerait jamais rien serait de 99,7 %, et ne dirait rien. Chaque machine connaît de 1 à 8 pannes.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Pannes_Parc.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : ranger le temps et mettre le test de côté

On trie les relevés de chaque machine par date. Le dernier quart de chaque série est mis de côté avant tout apprentissage : il joue le rôle de l'avenir, à une réserve près. La cible d'un relevé regarde 24 heures plus loin : les 24 derniers relevés de chaque machine n'ont pas d'avenir observé, leur cible vaudrait 0 même si une panne suit, et ils sont écartés du test. Pour la même raison, l'apprentissage ne garde pas les 24 derniers relevés qui précèdent la coupure de chaque machine : leur cible regarde jusque dans le test. Aucun mélange des lignes n'est permis.

In [2]:
HORIZON = 24                                                    # heures
df = df.sort_values(["machine", "horodatage"])
effectif = df.groupby("machine")["horodatage"].transform("size")
rang_depuis_la_fin = df.groupby("machine").cumcount(ascending=False)    # 0 : dernier relevé de la machine
taille_test = np.ceil(0.25 * effectif)
est_test = (rang_depuis_la_fin < taille_test) & (rang_depuis_la_fin >= HORIZON)   # sans les 24 derniers relevés
est_train = rang_depuis_la_fin >= taille_test + HORIZON
print("apprentissage :", int(est_train.sum()), "relevés ; test :", int(est_test.sum()), "relevés ; écartés :", int((~est_train & ~est_test).sum()))

apprentissage : 17616 relevés ; test : 5616 relevés ; écartés : 768


### Étape 3 : construire la cible et les variables

La cible d'un relevé vaut 1 si l'une des pannes des 24 relevés suivants de la même machine vaut 1. Les 24 derniers relevés d'une machine n'ont pas 24 relevés devant eux : leur cible reçoit 0, et ils ne servent ni à l'apprentissage ni au test. Les variables ne regardent que le passé de la machine : les mesures du relevé, et un indicateur qui vaut 1 si une panne s'est produite pendant les vingt-quatre heures écoulées, relevé courant compris.

In [3]:
par_machine = df.groupby("machine")
panne = df["panne"]
a_venir = pd.concat([par_machine["panne"].shift(-pas) for pas in range(1, HORIZON + 1)], axis=1).max(axis=1)
y = a_venir.fillna(0).astype(int)                                       # 1 : une panne dans les 24 heures suivantes
mesures = ["charge_pct", "temperature_C", "vibration_mm_s", "heures_depuis_revision"]
X = df[mesures].assign(panne_24h_passees=par_machine["panne"].transform(lambda s: s.rolling(HORIZON, min_periods=1).max()))
X_train, X_test = X[est_train], X[est_test]
y_train, y_test = y[est_train], y[est_test]
print("cible positive : apprentissage", int(y_train.sum()), "sur", len(y_train), "; test", int(y_test.sum()), "sur", len(y_test))
print("pannes dans la base de test :", int(panne[est_test].sum()))

cible positive : apprentissage 1152 sur 17616 ; test 479 sur 5616
pannes dans la base de test : 18


Lecture du résultat : la cible « panne dans les vingt-quatre heures » vaut 1 pour 6,5 % des relevés d'apprentissage et 8,5 % de ceux du test : une panne rend positifs les 24 relevés qui la précèdent. La base de test compte 479 relevés positifs et 18 pannes, de quoi juger la classe rare.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression logistique sur les quatre mesures et sur l'indicateur de panne récente. Les cases vides sont remplacées par la médiane de l'apprentissage, les variables sont mises à l'échelle, et `class_weight="balanced"` donne à la classe rare un poids proportionnel à sa rareté : sans lui, la régression logistique répondrait toujours « pas de panne ». Le pipeline apprend sur `X_train` seul.

In [4]:
# Solution :
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

colonnes_depart = mesures + ["panne_24h_passees"]
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(class_weight="balanced", max_iter=1000))
model.fit(X_train[colonnes_depart], y_train)
# Fin de la solution

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

### Étape 5 : mesurer sur la base de test

Le modèle prédit des relevés qu'il n'a jamais vus. On lit le F-score macro, la précision (part des alertes justifiées) et le rappel (part des pannes annoncées) de la classe rare, et la matrice de confusion. Quatre comparaisons sans apprentissage : ne jamais signaler, la persistance (signaler tant qu'une panne a eu lieu dans les vingt-quatre heures écoulées), et les deux règles fixées à la main, la vibration au-dessus de 8 mm/s et la révision de plus de 240 heures.

In [5]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

y_pred = model.predict(X_test[colonnes_depart])
score_depart = f1_score(y_test, y_pred, average="macro")
seuil_main = 8.0                                                # mm/s, fixé à la main
score_jamais = f1_score(y_test, np.zeros(len(y_test), dtype=int), average="macro")
score_persistance = f1_score(y_test, X_test["panne_24h_passees"].astype(int), average="macro")
score_vibration = f1_score(y_test, (X_test["vibration_mm_s"] > seuil_main).astype(int), average="macro")
score_revision = f1_score(y_test, (X_test["heures_depuis_revision"] >= 240).astype(int), average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"précision de la classe rare : {precision_score(y_test, y_pred):.3f}, rappel : {recall_score(y_test, y_pred):.3f}")
print(f"F-score macro, ne jamais signaler : {score_jamais:.3f}")
print(f"F-score macro, persistance : {score_persistance:.3f}")
print(f"F-score macro, vibration au-dessus de {seuil_main:.0f} mm/s : {score_vibration:.3f}")
print(f"F-score macro, révision de plus de 240 heures : {score_revision:.3f}")
print("matrice de confusion (lignes : réel, colonnes : prédit ; pas de panne, panne)")
print(confusion_matrix(y_test, y_pred))

F-score macro : 0.700
précision de la classe rare : 0.340, rappel : 0.900
F-score macro, ne jamais signaler : 0.478
F-score macro, persistance : 0.458
F-score macro, vibration au-dessus de 8 mm/s : 0.695
F-score macro, révision de plus de 240 heures : 0.564
matrice de confusion (lignes : réel, colonnes : prédit ; pas de panne, panne)
[[4302  835]
 [  48  431]]


Lecture du résultat : le F-score macro est de 0,700, contre 0,478 pour le témoin qui ne signale jamais rien. Le rappel de la classe rare est de 0,900 pour une précision de 0,340. Les règles à la main font 0,695 pour la vibration et 0,564 pour la révision. La vibration d'une machine saine diffère d'une machine à l'autre, au plus bas de 1,0 à 2,3 mm/s : un seuil unique ne convient pas à toutes.

## Pour aller plus loin : comparer à la machine elle-même, et valider dans le temps

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les trois cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première construit des variables qui comparent chaque machine à elle-même : la vibration rapportée à la plus basse vibration déjà observée sur cette machine, ses moyennes sur les 6 et les 24 dernières heures, et leur écart (la tendance). La deuxième met en concurrence, par validation glissante sur l'apprentissage seul (`TimeSeriesSplit`, quatre découpages successifs, toujours entraînés sur le passé, avec un trou de 24 heures de seize machines entre apprentissage et validation pour que la cible ne lise pas l'avenir de la validation), une petite grille : la régression logistique sur ces variables, pour quatre poids de la classe rare et trois réglages de la régularisation, un arbre de profondeur 1, un arbre de profondeur 3, une forêt aléatoire et un gradient boosting. Elle note aussi la règle de vibration, avec un seuil choisi en validation. La troisième mesure le modèle retenu, et cette règle, une seule fois sur le test. Les lectures sont imprimées par les cellules elles-mêmes.

In [6]:
# Solution :
vibration = par_machine["vibration_mm_s"]
X_plus = X.assign(vibration_relative=df["vibration_mm_s"] / vibration.transform(lambda s: s.expanding().min()),
                  vibration_moy6=vibration.transform(lambda s: s.rolling(6, min_periods=1).mean()),
                  vibration_moy24=vibration.transform(lambda s: s.rolling(24, min_periods=1).mean()))
X_plus["variation_vibration"] = X_plus["vibration_moy6"] - X_plus["vibration_moy24"]
toutes = list(X_plus.columns)
vr = X_plus.loc[X_train.index, "vibration_relative"]
print(f"{len(toutes)} variables ; vibration relative de l'apprentissage : minimum {vr.min():.2f}, médiane {vr.median():.2f}, maximum {vr.max():.2f}")
# Fin de la solution

9 variables ; vibration relative de l'apprentissage : minimum 1.00, médiane 2.28, maximum 7.33


In [7]:
# Solution :
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeClassifier

mediane = lambda estimateur: make_pipeline(SimpleImputer(strategy="median"), estimateur)
candidats = {}
for poids in (1, 2, 3, 5):                                      # poids de la classe rare ; 1 : sans pondération
    for C in (0.3, 1, 3):                                       # inverse de la régularisation
        logit = LogisticRegression(class_weight={0: 1, 1: poids}, C=C, max_iter=1000)
        candidats[f"logistique, poids {poids}, C {C}"] = (make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), logit), toutes)
candidats["arbre de profondeur 1 sur la vibration (un seuil)"] = (
    mediane(DecisionTreeClassifier(max_depth=1, class_weight="balanced", random_state=GRAINE)), ["vibration_mm_s"])
candidats["arbre de profondeur 3"] = (
    mediane(DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=GRAINE)), toutes)
candidats["forêt aléatoire"] = (
    mediane(RandomForestClassifier(class_weight="balanced", min_samples_leaf=5, random_state=GRAINE)), toutes)
candidats["gradient boosting par histogrammes"] = (
    mediane(HistGradientBoostingClassifier(class_weight="balanced", random_state=GRAINE)), toutes)

ordre = df.loc[X_train.index, "horodatage"].argsort(kind="stable")      # lignes d'apprentissage dans l'ordre du temps
X_ordre, y_ordre = X_plus.loc[X_train.index].iloc[ordre], y_train.iloc[ordre]
cv = TimeSeriesSplit(n_splits=4, gap=HORIZON * df["machine"].nunique())  # trou : 24 heures de toutes les machines
notes_cv = {}
for nom, (estimateur, colonnes) in candidats.items():
    notes = cross_val_score(estimateur, X_ordre[colonnes], y_ordre, cv=cv, scoring="f1_macro")
    notes_cv[nom] = notes.mean()
    print(f"{nom} : F-score macro {notes.mean():.3f} +/- {notes.std():.3f}")
nom_retenu = max(notes_cv, key=notes_cv.get)
print("modèle retenu par la validation glissante :", nom_retenu)

regles = {}                                                     # règle sans apprentissage : seuil de vibration, de 6 à 13 mm/s
for seuil in range(6, 14):
    pred = (X_ordre["vibration_mm_s"] > seuil).astype(int)
    regles[seuil] = np.mean([f1_score(y_ordre.iloc[v], pred.iloc[v], average="macro") for _, v in cv.split(X_ordre)])
seuil_valide = max(regles, key=regles.get)
print(f"règle de vibration, seuil choisi en validation : {seuil_valide} mm/s, F-score macro {regles[seuil_valide]:.3f}")
# Fin de la solution

logistique, poids 1, C 0.3 : F-score macro 0.818 +/- 0.027
logistique, poids 1, C 1 : F-score macro 0.815 +/- 0.038
logistique, poids 1, C 3 : F-score macro 0.815 +/- 0.047
logistique, poids 2, C 0.3 : F-score macro 0.838 +/- 0.026
logistique, poids 2, C 1 : F-score macro 0.828 +/- 0.041
logistique, poids 2, C 3 : F-score macro 0.823 +/- 0.051
logistique, poids 3, C 0.3 : F-score macro 0.829 +/- 0.029
logistique, poids 3, C 1 : F-score macro 0.822 +/- 0.039
logistique, poids 3, C 3 : F-score macro 0.820 +/- 0.047
logistique, poids 5, C 0.3 : F-score macro 0.817 +/- 0.028
logistique, poids 5, C 1 : F-score macro 0.812 +/- 0.034
logistique, poids 5, C 3 : F-score macro 0.810 +/- 0.041
arbre de profondeur 1 sur la vibration (un seuil) : F-score macro 0.640 +/- 0.038
arbre de profondeur 3 : F-score macro 0.697 +/- 0.074
forêt aléatoire : F-score macro 0.806 +/- 0.060
gradient boosting par histogrammes : F-score macro 0.741 +/- 0.117
modèle retenu par la validation glissante : logistique, p

In [8]:
# Solution :
estimateur, colonnes = candidats[nom_retenu]
estimateur.fit(X_plus.loc[X_train.index, colonnes], y_train)
pred_retenu = estimateur.predict(X_plus.loc[X_test.index, colonnes])
f1_retenu = f1_score(y_test, pred_retenu, average="macro")
print(f"{nom_retenu}, mesuré une fois sur le test : F-score macro {f1_retenu:.3f}")
print(f"précision {precision_score(y_test, pred_retenu):.3f}, rappel {recall_score(y_test, pred_retenu):.3f}")
print(f"gain sur le modèle de départ : {f1_retenu - score_depart:+.3f}")
f1_regle = f1_score(y_test, (X_test["vibration_mm_s"] > seuil_valide).astype(int), average="macro")
print(f"règle de vibration à {seuil_valide} mm/s, seuil choisi en validation, mesurée une fois sur le test : F-score macro {f1_regle:.3f}")
print(f"gain sur la règle : {f1_retenu - f1_regle:+.3f}")
# Fin de la solution

logistique, poids 2, C 0.3, mesuré une fois sur le test : F-score macro 0.817
précision 0.592, rappel 0.775
gain sur le modèle de départ : +0.117
règle de vibration à 9 mm/s, seuil choisi en validation, mesurée une fois sur le test : F-score macro 0.735
gain sur la règle : +0.082


## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [9]:
import hashlib

score = f1_score(y_test, model.predict(X_test[colonnes_depart]), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

EMPREINTE_TEST = 7726380df65ca90b4e92f56f5d372eb2e2bd16965d5b25a9b4ec1768195529b8
SCORE_REFERENCE f1_macro = 0.700


## Synthèse

1. Prévoir une panne, c'est décaler la cible : le relevé courant reçoit la panne des 24 relevés suivants de sa machine, l'apprentissage s'arrête 24 relevés avant la coupure de chaque machine pour que sa cible ne lise pas le test, et le test s'arrête 24 relevés avant la fin de chaque machine, où l'avenir n'est pas observé.
2. Sur une classe rare, l'exactitude ne dit rien : le témoin qui ne signale jamais rien donne 0,478 de F-score macro. Le modèle de départ fait 0,700, une alarme à vibration fixe 0,695.
3. Un seuil unique ne convient pas à un parc : chaque machine a sa vibration saine. Rapporter chaque mesure à la machine elle-même, et regarder sa tendance, change la prévision : en validation glissante, le meilleur modèle de la grille atteint 0,838, la forêt 0,806, le gradient boosting 0,741, contre 0,724 pour la meilleure règle de vibration (9 mm/s) et 0,640 pour un seuil de vibration appris.
4. La validation glissante entraîne toujours sur le passé et valide sur l'avenir proche ; la validation croisée habituelle mélangerait le temps.
5. Le modèle retenu par la validation glissante est « logistique, poids 2, C 0.3 » ; mesuré une seule fois sur le test, il donne 0,817, contre 0,735 pour la règle de vibration dont le seuil est lui aussi choisi en validation. Une alerte reste une alerte : seule une inspection qualifie la machine.